# Pertemuan 7 — Praktikum 6: Pencarian Lagu Mirip pada Dataset Spotify

Praktikum ini menerapkan ketiga pustaka ANN pada dataset nyata berukuran besar, yaitu dataset lagu Spotify yang tersedia pada tautan Kaggle di modul. Dataset tersebut memuat dua berkas, dan berkas pertama dipakai pada praktikum ini karena memuat atribut audio lagu yang diperlukan.

Atribut yang dipakai adalah sembilan fitur numerik, yaitu *danceability*, *energy*, *loudness*, *speechiness*, *acousticness*, *instrumentalness*, *liveness*, *valence*, dan *tempo*. Fitur tersebut dinormalisasi dengan `StandardScaler` agar setiap atribut memberikan kontribusi yang seimbang pada perhitungan jarak.

Pencarian dilakukan untuk menemukan sepuluh lagu yang paling mirip dengan setiap lagu dalam dataset, kemudian hasil keempat metode dibandingkan.

Sumber: [Praktikum 6](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js06-ann-approximate-nearest-neighbors/praktikum-6).

## 1. Menyiapkan pustaka dan dataset

Berkas `spotify_songs.csv` memuat kolom identitas lagu beserta sembilan atribut audio hasil ekstraksi dari berkas pertama pada dataset Kaggle. Berkas aslinya berukuran sekitar 1,5 GB dan juga memuat kolom lirik yang tidak dipakai dalam perhitungan jarak.

In [1]:
%matplotlib inline

import time

import faiss
import hnswlib
import numpy as np
import pandas as pd
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [2]:
# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('spotify_songs.csv')

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

print("Jumlah lagu :", X_scaled.shape[0])
print("Jumlah fitur:", X_scaled.shape[1])
display(df.head())

Jumlah lagu : 955320
Jumlah fitur: 9


,id,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0Prct5TDjAnEgIqbxcldY9,!,HELLYEAH,0.415,0.6050,-11.157,0.0575,0.00116,0.838000,0.4710,0.193,100.059
1,2ASl4wirkeYm3OWZxXKYuq,!!,Yxngxr1,0.788,0.6480,-9.135,0.3150,0.90000,0.000000,0.1760,0.287,79.998
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Glowie,0.000,0.0354,-20.151,0.0000,0.90800,0.000000,0.4790,0.000,0.000
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Rosendo,0.657,0.8820,-6.340,0.0385,0.00740,0.000013,0.0474,0.939,123.588
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Rosendo,0.659,0.8930,-8.531,0.0411,0.09220,0.000019,0.0534,0.951,123.600


## 2. Exact Nearest Neighbor

Pencarian exact dilakukan dengan `NearestNeighbors` bermetode *brute force*. Hasilnya dipakai sebagai acuan untuk menilai kedekatan hasil ketiga pustaka ANN.

In [3]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean', n_jobs=-1)
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 508.717 s


## 3. Annoy

Annoy dibangun dengan 10 pohon. Karena jumlah lagu mendekati satu juta, nilai `search_k` ditetapkan secara eksplisit pada tahap pencarian. Nilai bawaan Annoy sebesar `n_trees × jumlah data` akan memeriksa sekitar sepuluh juta kandidat untuk setiap lagu, sehingga waktu pencariannya menjadi tidak praktis.

In [4]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k, search_k=2_000) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 175.892 s


## 4. HNSW

HNSW dibangun dengan `M = 16` dan `ef_construction = 200`, sedangkan pencarian memakai `ef = 200`.

In [5]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 48.959 s


## 5. FAISS IVF

FAISS memakai `IndexIVFFlat` dengan 100 klaster. Pada tahap pencarian, `nprobe = 10` membuat pencarian hanya dilakukan pada sepuluh klaster yang paling dekat dengan query.

In [6]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 144.121 s


## 6. Contoh hasil pencarian untuk lagu pertama

Sel berikut menampilkan lima tetangga terdekat dari lagu pertama menurut keempat metode, beserta nama lagu dan artisnya. Penyajian nama lagu membantu menilai apakah lagu yang ditemukan memang memiliki karakteristik audio yang mirip.

In [7]:
# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("Top-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

lagu_acuan = df.iloc[0]
print(f"\nLagu acuan: {lagu_acuan['name']} - {lagu_acuan['artists']}")
print(f"Atribut: {lagu_acuan[features].to_dict()}")

for nama, idx in (("Exact NN", idx_exact[0][:5]), ("Annoy", idx_annoy[0][:5]),
                  ("HNSW", idx_hnsw[0][:5]), ("FAISS", idx_faiss[0][:5])):
    print(f"\n{nama}:")
    display(df.iloc[list(idx)][["name", "artists"] + features].reset_index(drop=True))

Top-5 neighbors for first song:
Exact NN: [     0 394553 764272 837727 749223]
Annoy:    [0, 394553, 764272, 837727, 749223]
HNSW:     [     0 394553 764272 837727 749223]
FAISS:    [     0 394553 764272 837727 749223]

Lagu acuan: ! - HELLYEAH
Atribut: {'danceability': 0.415, 'energy': 0.605, 'loudness': -11.157, 'speechiness': 0.0575, 'acousticness': 0.00116, 'instrumentalness': 0.838, 'liveness': 0.471, 'valence': 0.193, 'tempo': 100.059}

Exact NN:


,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,!,HELLYEAH,0.415,0.605,-11.157,0.0575,0.001160,0.838,0.471,0.1930,100.059
1,Intro (Who The Fuck Are You'),C-Lekktor,0.486,0.617,-13.075,0.0584,0.003900,0.815,0.426,0.0951,110.982
2,Testimonium,Lacrimosa,0.430,0.573,-8.996,0.0309,0.000008,0.768,0.367,0.1880,101.990
3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,0.412,0.723,-10.706,0.0354,0.026700,0.890,0.410,0.0354,98.003
4,Sæglópur,Sigur Rós,0.351,0.609,-9.031,0.0418,0.061400,0.803,0.386,0.1000,102.041



Annoy:


,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,!,HELLYEAH,0.415,0.605,-11.157,0.0575,0.001160,0.838,0.471,0.1930,100.059
1,Intro (Who The Fuck Are You'),C-Lekktor,0.486,0.617,-13.075,0.0584,0.003900,0.815,0.426,0.0951,110.982
2,Testimonium,Lacrimosa,0.430,0.573,-8.996,0.0309,0.000008,0.768,0.367,0.1880,101.990
3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,0.412,0.723,-10.706,0.0354,0.026700,0.890,0.410,0.0354,98.003
4,Sæglópur,Sigur Rós,0.351,0.609,-9.031,0.0418,0.061400,0.803,0.386,0.1000,102.041



HNSW:


,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,!,HELLYEAH,0.415,0.605,-11.157,0.0575,0.001160,0.838,0.471,0.1930,100.059
1,Intro (Who The Fuck Are You'),C-Lekktor,0.486,0.617,-13.075,0.0584,0.003900,0.815,0.426,0.0951,110.982
2,Testimonium,Lacrimosa,0.430,0.573,-8.996,0.0309,0.000008,0.768,0.367,0.1880,101.990
3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,0.412,0.723,-10.706,0.0354,0.026700,0.890,0.410,0.0354,98.003
4,Sæglópur,Sigur Rós,0.351,0.609,-9.031,0.0418,0.061400,0.803,0.386,0.1000,102.041



FAISS:


,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,!,HELLYEAH,0.415,0.605,-11.157,0.0575,0.001160,0.838,0.471,0.1930,100.059
1,Intro (Who The Fuck Are You'),C-Lekktor,0.486,0.617,-13.075,0.0584,0.003900,0.815,0.426,0.0951,110.982
2,Testimonium,Lacrimosa,0.430,0.573,-8.996,0.0309,0.000008,0.768,0.367,0.1880,101.990
3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,0.412,0.723,-10.706,0.0354,0.026700,0.890,0.410,0.0354,98.003
4,Sæglópur,Sigur Rós,0.351,0.609,-9.031,0.0418,0.061400,0.803,0.386,0.1000,102.041


## 7. Perbandingan waktu dan tingkat kesesuaian

Sel berikut merangkum waktu setiap metode dan menghitung *recall* terhadap hasil exact, yaitu proporsi tetangga yang sama dengan hasil pencarian exact untuk seluruh lagu.

In [8]:
def recall_at_k(I_pred, I_gt, k):
    correct = 0
    for i in range(len(I_pred)):
        correct += len(set(np.asarray(I_pred[i])[:k]) & set(np.asarray(I_gt[i])[:k]))
    return correct / (len(I_pred) * k)


ringkasan = pd.DataFrame([
    {"Metode": "Exact NN (brute force)", "Waktu (detik)": round(time_exact, 3), "Recall@10": 1.0},
    {"Metode": "Annoy (10 trees)", "Waktu (detik)": round(time_annoy, 3),
     "Recall@10": round(recall_at_k(idx_annoy, idx_exact, k), 4)},
    {"Metode": "HNSW (ef=200)", "Waktu (detik)": round(time_hnsw, 3),
     "Recall@10": round(recall_at_k(idx_hnsw, idx_exact, k), 4)},
    {"Metode": "FAISS IVF (nprobe=10)", "Waktu (detik)": round(time_faiss, 3),
     "Recall@10": round(recall_at_k(idx_faiss, idx_exact, k), 4)},
])

display(ringkasan)

,Metode,Waktu (detik),Recall@10
0,Exact NN (brute force),508.717,1.0000
1,Annoy (10 trees),175.892,0.9976
2,HNSW (ef=200),48.959,0.9947
3,FAISS IVF (nprobe=10),144.121,0.9977


## 8. Analisis

Dataset Spotify berisi ratusan ribu lagu dengan sembilan atribut audio. Setiap atribut memiliki satuan dan rentang yang berbeda, misalnya `tempo` yang bernilai puluhan hingga ratusan dan `acousticness` yang bernilai antara 0 dan 1. Tanpa normalisasi, atribut dengan rentang besar akan mendominasi perhitungan jarak. `StandardScaler` menyetarakan skala seluruh atribut sehingga kedekatan yang diukur mencerminkan kemiripan karakteristik audio secara keseluruhan.

Pencarian exact menghitung jarak antara setiap lagu dengan seluruh lagu lain. Jumlah perhitungan ini bertambah secara kuadratik terhadap jumlah lagu, sehingga waktu komputasinya paling lama di antara keempat metode. Ketiga metode ANN membangun struktur tambahan di awal, kemudian memakai struktur tersebut untuk mempersempit ruang pencarian.

Pada hasil pencarian untuk lagu pertama, keempat metode mengembalikan lima lagu yang sama beserta urutan yang sama. Hal ini terjadi karena lagu acuan berada pada bagian data yang mudah dipisahkan, sehingga kandidat yang diperiksa sudah mencakup tetangga yang sebenarnya. Secara keseluruhan, recall ketiga metode berada di atas 0,99, yang berarti lebih dari 99% tetangga yang ditemukan sama dengan hasil pencarian exact. Perbedaan yang tetap muncul pada sebagian lagu lain umumnya berupa pergeseran urutan atau pergantian satu hingga dua tetangga pada posisi akhir, dan hal ini wajar karena ANN hanya memeriksa sebagian kandidat, bukan seluruh data.

Nilai recall memperlihatkan seberapa besar perbedaan tersebut. Ketiga metode menghasilkan recall di atas 0,99, dengan FAISS IVF sedikit lebih tinggi (0,9977), diikuti Annoy (0,9976) dan HNSW (0,9947). Nilai tersebut menunjukkan bahwa pada dataset ini ketiga metode hampir selalu menemukan tetangga yang sama dengan pencarian exact. Waktu pencarian HNSW paling singkat, diikuti FAISS IVF, lalu Annoy, sedangkan pencarian exact memerlukan waktu paling lama. Nilai recall FAISS maupun HNSW dapat dinaikkan lagi dengan memperbesar `nprobe` atau `ef`, dengan konsekuensi waktu pencarian bertambah.

## Kesimpulan

Ketiga pustaka ANN mampu melakukan pencarian lagu mirip pada dataset berukuran ratusan ribu dengan waktu yang jauh lebih singkat daripada pencarian exact. Normalisasi fitur menjadi langkah penting karena atribut audio memiliki rentang nilai yang berbeda. HNSW memberikan keseimbangan terbaik antara recall dan waktu pada percobaan ini karena menghasilkan recall 0,9947 dengan waktu pencarian paling singkat. FAISS IVF dan Annoy menghasilkan recall yang sedikit lebih tinggi, yaitu 0,9977 dan 0,9976, dengan waktu pencarian yang lebih lama. Seluruh metode ANN menghasilkan tetangga yang hampir sama dengan hasil exact, sehingga pencarian yang mendekati hasil sebenarnya dapat dilakukan jauh lebih cepat daripada pencarian exact.